# 🧩 Mini-Lab: Building a CLI

**Module 10: Delivery: CLI, API, and Debugging Lab 2** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Choose** CLI as the required delivery interface (HTTP is optional later)
2. **Build** an `argparse` CLI with `--help`, exit codes, and stdout vs stderr
3. **Exercise** the shipped entry point with 1–2 subprocess integration tests

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Command-line interface | Arguments, `--help`, exit codes, stdout versus stderr |
| Integration test | Exercise the shipped entry point, not just internal functions |
| Workflow form choice | Notebook, script, package, or service |


## Why CLI Conventions Matter

A CLI that follows Unix conventions is **composable** — it can be piped, scripted, and tested by other tools. The three pillars are:

| Pillar | Convention | Why |
|--------|-----------|-----|
| **Exit codes** | `0` = success, non-zero = failure | Scripts and CI check `$?` / `%ERRORLEVEL%` to decide what to do next |
| **stdout vs stderr** | Results → `stdout`, logs/errors → `stderr` | So `cmd | jq .` works — pipes only carry stdout |
| **Help text** | `--help` / `-h` prints usage and exits | Users discover arguments without reading source code |

---
## Which interface? CLI is the required path

HTTP/FastAPI is optional. Decide CLI here.


## Why Interface Choice Matters

You've built the **core logic** — maybe a summariser, an extractor, or a RAG pipeline. Now you need to let someone (or something) **use** it. The interface you wrap around your core determines:

- **Who** can call it (a developer in a terminal? a frontend app? a cron job?)
- **How** results flow (stdout, JSON over HTTP, exit codes)
- **Where** it runs (laptop, CI, cloud server)

Picking the wrong form wastes effort. Picking the right one can be a one-afternoon task.

## The Three Interface Forms

| Form | Entry Point | Typical Consumer | Best For |
|------|-------------|------------------|----------|
| **CLI** | `python main.py --arg value` | Developer, script, cron, CI | Batch jobs, automation, local tools |
| **HTTP API** | `POST /api/summarise` | Frontend, mobile app, other services | Real-time requests, multi-client access |
| **Both** | CLI *and* HTTP share the same core | Mixed audiences | Projects that start local then go online |

## A Decision Framework

We'll encode the tradeoffs into a simple scoring model. For each scenario you answer three questions, and the scores point you toward CLI, HTTP, or both.

In [2]:
# Decision dimensions and how each interface scores on them.
# Scores are illustrative (1 = weak fit, 3 = strong fit).

DIMENSIONS = {
    "human_at_terminal": {
        "question": "Is the primary user a developer or ops person at a terminal?",
        "scores": {"CLI": 3, "HTTP": 1, "Both": 2},
    },
    "called_by_other_services": {
        "question": "Will other services (frontend, mobile, microservices) call it over the network?",
        "scores": {"CLI": 1, "HTTP": 3, "Both": 3},
    },
    "batch_or_pipeline": {
        "question": "Does it run as a batch/pipeline step (cron, CI, scripts)?",
        "scores": {"CLI": 3, "HTTP": 1, "Both": 2},
    },
    "multiple_clients_concurrently": {
        "question": "Do multiple clients need to hit it at the same time?",
        "scores": {"CLI": 1, "HTTP": 3, "Both": 3},
    },
    "quick_local_iteration": {
        "question": "Is fast local iteration (edit → run → check) a priority?",
        "scores": {"CLI": 3, "HTTP": 2, "Both": 2},
    },
}

In [3]:
def recommend_interface(answers: dict[str, bool]) -> str:
    """Given yes/no answers for each dimension, return a recommendation.

    Args:
        answers: Maps dimension key -> True (yes) or False (no).

    Returns:
        Markdown-formatted recommendation string.
    """
    totals = {"CLI": 0, "HTTP": 0, "Both": 0}

    for dim_key, is_yes in answers.items():
        if not is_yes:
            continue  # Only accumulate score for "yes" answers
        scores = DIMENSIONS[dim_key]["scores"]
        for form, score in scores.items():
            totals[form] += score

    ranked = sorted(totals.items(), key=lambda x: x[1], reverse=True)
    winner = ranked[0][0]

    lines = ["### Scores\n"]
    lines.append("| Interface | Score |")
    lines.append("|-----------|-------|")
    for form, score in ranked:
        marker = " ⬅️ recommended" if form == winner else ""
        lines.append(f"| **{form}** | {score}{marker} |")

    lines.append(f"\n**Recommendation → {winner}**")
    return "\n".join(lines)

## Scenario 1 — Nightly Report Summariser

> *A script runs every night via cron, reads new documents from a folder, summarises them, and writes a report file. No web users.*

In [4]:
scenario_1 = {
    "human_at_terminal":           True,   # Dev runs & debugs it locally
    "called_by_other_services":     False,  # No network consumers
    "batch_or_pipeline":            True,   # Runs via cron every night
    "multiple_clients_concurrently": False, # Single scheduled job
    "quick_local_iteration":        True,   # Developer iterates on prompts
}

md(recommend_interface(scenario_1))

### Scores

| Interface | Score |
|-----------|-------|
| **CLI** | 9 ⬅️ recommended |
| **Both** | 6 |
| **HTTP** | 4 |

**Recommendation → CLI**

CLI wins here: the consumer is a cron job and a developer. No web traffic, no concurrent clients.

## Scenario 2 — Customer Support Chat Widget

> *A React frontend sends user messages to a backend that calls an LLM and streams answers back. Hundreds of users may be active simultaneously.*

In [5]:
scenario_2 = {
    "human_at_terminal":           False,  # End users are on a web page
    "called_by_other_services":     True,   # React frontend calls API
    "batch_or_pipeline":            False,  # Real-time, not batch
    "multiple_clients_concurrently": True,  # Hundreds of users at once
    "quick_local_iteration":        False,  # Iteration matters less than uptime
}

md(recommend_interface(scenario_2))

### Scores

| Interface | Score |
|-----------|-------|
| **HTTP** | 6 ⬅️ recommended |
| **Both** | 6 |
| **CLI** | 2 |

**Recommendation → HTTP**

HTTP wins: the consumer is a web frontend, concurrency is a must, and there's no terminal user.

## Scenario 3 — Internal RAG Tool

> *An engineering team queries a knowledge base. Some use a terminal; others use an internal web dashboard. A CI job also tests retrieval quality nightly.*

In [6]:
scenario_3 = {
    "human_at_terminal":           True,   # Engineers at terminal
    "called_by_other_services":     True,   # Web dashboard calls API
    "batch_or_pipeline":            True,   # CI quality checks
    "multiple_clients_concurrently": True,  # Dashboard has several users
    "quick_local_iteration":        True,   # Engineers iterate on queries
}

md(recommend_interface(scenario_3))

### Scores

| Interface | Score |
|-----------|-------|
| **Both** | 12 ⬅️ recommended |
| **CLI** | 11 |
| **HTTP** | 10 |

**Recommendation → Both**

"Both" wins: the same core logic needs to serve a terminal user, a web dashboard, **and** a CI pipeline. The key architecture insight is:

```
┌─────────┐     ┌────────────┐
│  CLI    │────▶│            │
└─────────┘     │  Core      │
┌─────────┐     │  Logic     │
│  HTTP   │────▶│ (shared)   │
└─────────┘     └────────────┘
```

Both the CLI and the HTTP endpoint import and call the **same core function**. The interface is just a thin wrapper — it parses input, calls the core, and formats the output for its medium.

## The Key Principle: Separate Core from Interface

No matter which form you choose, the decision is easy if your **core logic is a plain function** that knows nothing about `argparse` or `FastAPI`.

```python
# core.py  — no CLI or HTTP imports here
def summarise(text: str, max_sentences: int = 3) -> str:
    """Pure business logic."""
    ...
```

```python
# cli.py  — thin CLI wrapper
from core import summarise
import argparse, sys

parser = argparse.ArgumentParser()
parser.add_argument("text")
args = parser.parse_args()
print(summarise(args.text))
```

```python
# api.py  — thin HTTP wrapper
from core import summarise
from fastapi import FastAPI

app = FastAPI()

@app.post("/summarise")
def endpoint(text: str):
    return {"summary": summarise(text)}
```

This separation means your choice of interface is **reversible** — you can always add the other form later with minimal effort.

## Quick-Reference Cheat Sheet

In [7]:
cheat_sheet = """
| Signal | Start with |
|--------|------------|
| Only developers/scripts will use it | **CLI** |
| A web or mobile frontend needs it | **HTTP** |
| It runs in CI / cron / pipelines | **CLI** |
| Multiple concurrent users | **HTTP** |
| Both terminal users AND web consumers | **Both** (shared core) |
| You're unsure and it's early | **CLI first** — easiest to add HTTP later |
"""

md(cheat_sheet)


| Signal | Start with |
|--------|------------|
| Only developers/scripts will use it | **CLI** |
| A web or mobile frontend needs it | **HTTP** |
| It runs in CI / cron / pipelines | **CLI** |
| Multiple concurrent users | **HTTP** |
| Both terminal users AND web consumers | **Both** (shared core) |
| You're unsure and it's early | **CLI first** — easiest to add HTTP later |


## Setup

We'll write Python scripts to disk using `%%writefile` and then run them with `!python` (or `subprocess`). This simulates the real workflow: **write a `.py` file, run it from the terminal**.

No pip installs needed — `argparse`, `os`, `sys`, and `subprocess` are all in the standard library.

In [1]:
import subprocess, sys, os
from pathlib import Path

from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

def run(cmd, env=None):
    """Run a shell command and print stdout, stderr, and exit code."""
    merged_env = {**os.environ, **(env or {})}
    result = subprocess.run(
        cmd, shell=True, capture_output=True, text=True, env=merged_env
    )
    if result.stdout:
        print("[stdout]", result.stdout, end="")
    if result.stderr:
        print("[stderr]", result.stderr, end="")
    print(f"[exit code] {result.returncode}")
    return result

## Step 1 — A Minimal CLI with `argparse`

Every CLI starts the same way: create a parser, add arguments, and call `parse_args()`. `argparse` gives you `--help` for free.

In [2]:
%%writefile greet.py
"""A tiny CLI that greets the user."""
import argparse
import sys


def main():
    parser = argparse.ArgumentParser(
        description="Greet someone by name."
    )
    parser.add_argument("name", help="Name of the person to greet")
    parser.add_argument(
        "--loud", action="store_true", help="SHOUT the greeting"
    )

    args = parser.parse_args()

    greeting = f"Hello, {args.name}!"
    if args.loud:
        greeting = greeting.upper()

    # Results go to stdout
    print(greeting)


if __name__ == "__main__":
    main()

Writing greet.py


In [3]:
# Check the auto-generated help text
run(f"{sys.executable} greet.py --help")

[stdout] usage: greet.py [-h] [--loud] name

Greet someone by name.

positional arguments:
  name        Name of the person to greet

options:
  -h, --help  show this help message and exit
  --loud      SHOUT the greeting
[exit code] 0


CompletedProcess(args='C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe greet.py --help', returncode=0, stdout='usage: greet.py [-h] [--loud] name\n\nGreet someone by name.\n\npositional arguments:\n  name        Name of the person to greet\n\noptions:\n  -h, --help  show this help message and exit\n  --loud      SHOUT the greeting\n', stderr='')

In [4]:
# Normal run — result on stdout, exit code 0
run(f"{sys.executable} greet.py Alice")

[stdout] Hello, Alice!
[exit code] 0


CompletedProcess(args='C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe greet.py Alice', returncode=0, stdout='Hello, Alice!\n', stderr='')

**Key observations:**
- `argparse` generated `--help` automatically from our `description` and `help` strings.
- The greeting appeared on **stdout** and the exit code was **0** (success).
- If you forget the required `name` argument, `argparse` prints an error to **stderr** and exits with code **2**.

In [5]:
# Missing required argument → stderr + exit code 2
run(f"{sys.executable} greet.py")

[stderr] usage: greet.py [-h] [--loud] name
greet.py: error: the following arguments are required: name
[exit code] 2


CompletedProcess(args='C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe greet.py', returncode=2, stdout='', stderr='usage: greet.py [-h] [--loud] name\ngreet.py: error: the following arguments are required: name\n')

## Step 2 — Exit Codes and stderr

Your own error handling should follow the same pattern:
- **Errors/diagnostics → `stderr`** (so they don't pollute piped output)
- **Non-zero exit code** signals failure to the calling script or CI

Let's build a slightly more realistic CLI: a word counter that reads a file.

In [6]:
%%writefile wordcount.py
"""Count words in a text file."""
import argparse
import sys
from pathlib import Path


def main():
    parser = argparse.ArgumentParser(
        description="Count words in a text file."
    )
    parser.add_argument("file", help="Path to the text file")
    args = parser.parse_args()

    path = Path(args.file)

    # --- Error handling: stderr + non-zero exit ---
    if not path.exists():
        print(f"error: file not found: {path}", file=sys.stderr)
        sys.exit(1)

    text = path.read_text(encoding="utf-8")
    word_count = len(text.split())

    # Diagnostics → stderr (so they don't break pipes)
    print(f"Reading {path.name}...", file=sys.stderr)

    # Result → stdout (pipeable)
    print(word_count)


if __name__ == "__main__":
    main()

Writing wordcount.py


In [7]:
# Create a small test file
Path("sample.txt").write_text(
    "The quick brown fox jumps over the lazy dog", encoding="utf-8"
)

# Success: word count on stdout, log on stderr, exit 0
run(f"{sys.executable} wordcount.py sample.txt")

[stdout] 9
[stderr] Reading sample.txt...
[exit code] 0


CompletedProcess(args='C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe wordcount.py sample.txt', returncode=0, stdout='9\n', stderr='Reading sample.txt...\n')

In [8]:
# Failure: error on stderr, exit code 1
run(f"{sys.executable} wordcount.py missing.txt")

[stderr] error: file not found: missing.txt
[exit code] 1


CompletedProcess(args='C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe wordcount.py missing.txt', returncode=1, stdout='', stderr='error: file not found: missing.txt\n')

Notice the pattern:

| Situation | Stream | Exit Code |
|-----------|--------|-----------|
| Success — the word count | `stdout` | `0` |
| Progress log ("Reading...") | `stderr` | — |
| File not found | `stderr` | `1` |

If another script does `count=$(python wordcount.py data.txt)`, the variable `count` contains **only** the number — the log line stays on stderr and doesn't interfere.

## Step 3 — Layered Configuration

Real CLIs get settings from **multiple sources** with a clear priority:

```
CLI argument  (highest priority — explicit user intent)
      ↑
Env variable  (set once per session / in .env)
      ↑
Hardcoded default  (lowest priority — safe fallback)
```

This is **layered configuration**. Let's add it to our greeter.

In [9]:
%%writefile greet_v2.py
"""Greeter with layered configuration."""
import argparse
import os
import sys


# --- Defaults (lowest priority) ---
DEFAULT_GREETING = "Hello"


def resolve_greeting(cli_value: str | None) -> str:
    """Resolve greeting from CLI arg → env var → default."""
    if cli_value is not None:
        return cli_value                           # CLI wins
    env_value = os.environ.get("GREETING_WORD")
    if env_value is not None:
        return env_value                           # Env var second
    return DEFAULT_GREETING                        # Default last


def main():
    parser = argparse.ArgumentParser(
        description="Greet someone (supports layered config)."
    )
    parser.add_argument("name", help="Name of the person to greet")
    parser.add_argument(
        "--greeting",
        default=None,
        help="Greeting word (default: $GREETING_WORD or 'Hello')",
    )
    args = parser.parse_args()

    word = resolve_greeting(args.greeting)
    print(f"{word}, {args.name}!")


if __name__ == "__main__":
    main()

Writing greet_v2.py


In [10]:
# Layer 1: No CLI arg, no env var → uses default "Hello"
run(f"{sys.executable} greet_v2.py Alice")

[stdout] Hello, Alice!
[exit code] 0


CompletedProcess(args='C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe greet_v2.py Alice', returncode=0, stdout='Hello, Alice!\n', stderr='')

In [11]:
# Layer 2: Env var set, no CLI arg → uses env var
run(
    f"{sys.executable} greet_v2.py Alice",
    env={"GREETING_WORD": "Howdy"},
)

[stdout]

 Howdy, Alice!
[exit code] 0


CompletedProcess(args='C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe greet_v2.py Alice', returncode=0, stdout='Howdy, Alice!\n', stderr='')

In [12]:
# Layer 3: CLI arg provided → overrides everything
run(
    f"{sys.executable} greet_v2.py Alice --greeting Bonjour",
    env={"GREETING_WORD": "Howdy"},
)

[stdout] Bonjour, Alice!
[exit code] 0


CompletedProcess(args='C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe greet_v2.py Alice --greeting Bonjour', returncode=0, stdout='Bonjour, Alice!\n', stderr='')

The three runs demonstrate the priority chain in action:

| Run | CLI arg | Env var | Result | Source used |
|-----|---------|---------|--------|-------------|
| 1 | — | — | Hello, Alice! | Default |
| 2 | — | `Howdy` | Howdy, Alice! | Env var |
| 3 | `Bonjour` | `Howdy` | Bonjour, Alice! | CLI arg (highest priority) |

## Cleanup

In [13]:
# Remove the script files we created
for f in ["greet.py", "greet_v2.py", "wordcount.py", "sample.txt"]:
    Path(f).unlink(missing_ok=True)

print("Cleaned up.")

Cleaned up.


## Integration tests of the shipped entry point

The cells above already ran `greet.py` and `wordcount.py` through the OS — that is an **integration test**. A unit test would import `word_count()` and never touch argparse, exit codes, or stdout.

Two checks to keep: `--help` exits 0, and a missing required argument exits non-zero. Those are the tests that catch wiring bugs unit tests miss.

```python
import subprocess, sys

help_run = subprocess.run([sys.executable, "greet.py", "--help"], capture_output=True, text=True)
assert help_run.returncode == 0, help_run.stderr

bad = subprocess.run([sys.executable, "greet.py"], capture_output=True, text=True)
assert bad.returncode != 0
assert bad.stderr  # usage / error belongs on stderr
print("CLI integration checks passed")
```

HTTP/`TestClient` tests live in the **optional** FastAPI mini, not on the required path.


## 🎯 Summary

### Key Takeaways

1. **`argparse` gives you a CLI contract** — positional args, optional flags, and auto-generated `--help` with almost no boilerplate
2. **Exit codes signal success or failure** — `0` means success; any non-zero value tells scripts, CI, and cron that something went wrong
3. **stdout is for results, stderr is for everything else** — errors, progress logs, and diagnostics go to `stderr` so piped output stays clean
4. **Layered configuration (default → env var → CLI arg)** — lets users set defaults once in their environment while still overriding per-invocation from the command line